# Environment Smooth Exploration

In [ ]:
from nescafe23_fluctuations import tools
import xarray as xr
import pygenray as pr
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

In [ ]:
def construct_envi(envi_field):
    # shift front location
    front_location = 35e3  # m

    envi_field = envi_field.assign_coords({
        "range": (envi_field.range.values + front_location)
    })

    c = envi_field["c"]

    bathy = xr.DataArray(
        np.linspace(4908, 4908, 100),  # flat bottom
        dims=["range"],
        coords={"range": np.linspace(0, 200e3, 100)},
    )

    envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)

    return envi

In [ ]:
def compute_rays(r_smooth, z_smooth):
    '''
    Given the vertical and horizontal smoothing values,
        compute rays, eigenrays, and ray_seperations

    Parameters
    ----------
    r_smooth : float
    z_smooth : float

    Returns
    -------
    rays
    eigs
    xi1s
    '''

    field = tools.create_2D_ocean_slice(z_smooth, r_smooth)
    envi = construct_envi(field)

    source_depth = 2361.50872806
    source_range = 0
    launch_angles = np.linspace(-30, 30, 3000)
    receiver_range = 150404.34909747
    num_range_save = 1000
    receiver_depths = [999.37979522]

    rays = pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi)

    eigs= pr.find_eigenrays(
        rays,
        receiver_depths,
        source_depth,
        source_range,
        receiver_range,
        num_range_save,
        envi,
        ztol=10,
        max_iter=50,
    )

    eig_angles = eigs.launch_angles[0]
    eig_angles_hat = eig_angles + np.degrees(1e-8)

    eig_dtheta_fan = pr.shoot_rays(source_depth, source_range, eig_angles_hat, receiver_range, num_range_save, envi)

    xi1s = eigs.zs[0] - eig_dtheta_fan.zs
    xi1s = xi1s / xi1s[:,-1][:,None]

    return rays, eigs, xi1s

In [ ]:
depth_smooths = [100, 300, 500, 700, 900, 1100, 1100]
range_smooths = [3000, 3000, 3000, 3000, 3000, 3000, 15000]
rays = []
eigs = []
xi1s = []
for k in range(len(depth_smooths)):

    print(depth_smooths[k])
    ray, eig, xi1 = compute_rays(r_smooth=range_smooths[k], z_smooth=depth_smooths[k])

    rays.append(ray)
    eigs.append(eig)
    xi1s.append(xi1)


In [ ]:
rays[3].plot_time_front(c='C0', add_colorbar=False)
plt.xlim([100.5,101.25])

plt.ylim([-2000,0])

In [ ]:
for k in range(6):
    eig_idx = np.where(eigs[k].ray_id[0] == '-6.0')[0]
    plt.plot(rays[k].rs[0,:]/1000, xi1s[k][eig_idx,:].T, label=depth_smooths[k])

    if k == 3:
        break
plt.legend()

In [ ]:
for k in range(6):
    eig_idx = np.where(eigs[k].ray_id[0] == '6.0')[0]
    plt.plot(rays[k].rs[0,:]/1000, xi1s[k][eig_idx,:].T, label=depth_smooths[k])

    if k == 3:
        break
            
plt.legend()

In [ ]:
eig_idx = np.where(eigs.ray_id[0] == '-7.0')[0]
plt.plot(rays.rs[0,:]/1000, xi1s[eig_idx,:].T)

eig_idx = np.where(eigs.ray_id[0] == '7.0')[0]
plt.plot(rays.rs[0,:]/1000, xi1s[eig_idx,:].T)

eig_idx = np.where(eigs.ray_id[0] == '-6.0')[0]
plt.plot(rays.rs[0,:]/1000, xi1s[eig_idx,:].T, label='-6')

eig_idx = np.where(eigs.ray_id[0] == '6.0')[0]
plt.plot(rays.rs[0,:]/1000, xi1s[eig_idx,:].T)

plt.legend()

In [ ]:
eig_idx = np.where(eigs.ray_id[0] == '-7.0')[0]
print(np.max(eigs.zs[0][eig_idx[0],700:]))

plt.plot(eigs.zs[0][eig_idx[0],:].T)

eig_idx = np.where(eigs.ray_id[0] == '7.0')[0]
print(np.max(eigs.zs[0][eig_idx[0],700:]))
plt.plot(eigs.zs[0][eig_idx[0],:].T)

eig_idx = np.where(eigs.ray_id[0] == '-6.0')[0]
print(np.max(eigs.zs[0][eig_idx[0],700:]))
plt.plot(eigs.zs[0][eig_idx[0],:].T)

eig_idx = np.where(eigs.ray_id[0] == '6.0')[0]
print(np.max(eigs.zs[0][eig_idx[0],700:]))
plt.plot(eigs.zs[0][eig_idx[0],:].T)

plt.xlim([700,1000])

In [ ]:
field_z200_r3000 = tools.create_2D_ocean_slice(z_smooth = 200, r_smooth = 3000)
field_z300_r5000 = tools.create_2D_ocean_slice(z_smooth = 300, r_smooth = 5000)
field_z400_r5000 = tools.create_2D_ocean_slice(z_smooth = 400, r_smooth = 5000)
field_z500_r5000 = tools.create_2D_ocean_slice(z_smooth = 500, r_smooth = 5000)
field_z800_r10000 = tools.create_2D_ocean_slice(z_smooth = 500, r_smooth = 5000)

envi_z200_r3000 = construct_envi(field_z200_r3000)
envi_z300_r5000 = construct_envi(field_z300_r5000)
envi_z400_r5000 = construct_envi(field_z400_r5000)
envi_z500_r5000 = construct_envi(field_z500_r5000)
envi_z800_r10000 = construct_envi(field_z800_r10000)

In [ ]:
source_depth = 2361.50872806
source_range = 0
launch_angles = np.linspace(-30, 30, 3000)
receiver_range = 150404.34909747
num_range_save = 1000
receiver_depths = [999.37979522]

In [ ]:
rays_z200_r3000 = pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi_z200_r3000)

In [ ]:
rays_z300_r5000 = pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi_z300_r5000)

In [ ]:
rays_z400_r5000 = pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi_z400_r5000)

In [ ]:
rays_z500_r5000 = pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi_z500_r5000)

In [ ]:
rays_z800_r10000 = pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi_z800_r10000)

In [ ]:
eigs_z200_r3000 = pr.find_eigenrays(
    rays_z200_r3000,
    receiver_depths,
    source_depth,
    source_range,
    receiver_range,
    num_range_save,
    envi_z200_r3000,
    ztol=10,
    max_iter=50,
)

eigs_z300_r5000 = pr.find_eigenrays(
    rays_z300_r5000,
    receiver_depths,
    source_depth,
    source_range,
    receiver_range,
    num_range_save,
    envi_z300_r5000,
    ztol=10,
    max_iter=50,
)

eigs_z400_r5000 = pr.find_eigenrays(
    rays_z400_r5000,
    receiver_depths,
    source_depth,
    source_range,
    receiver_range,
    num_range_save,
    envi_z400_r5000,
    ztol=10,
    max_iter=50,
)

eigs_z500_r5000 = pr.find_eigenrays(
    rays_z500_r5000,
    receiver_depths,
    source_depth,
    source_range,
    receiver_range,
    num_range_save,
    envi_z500_r5000,
    ztol=10,
    max_iter=50,
)

eigs_z800_r10000 = pr.find_eigenrays(
    rays_z800_r10000,
    receiver_depths,
    source_depth,
    source_range,
    receiver_range,
    num_range_save,
    envi_z800_r10000,
    ztol=10,
    max_iter=50,
)

In [ ]:
field_z800_r10000['c'].differentiate('depth').plot()

In [ ]:
eig_angles = eigs_z200_r3000.launch_angles[0]
eig_angles_hat = eig_angles + np.degrees(1e-5)

In [ ]:
eig_dtheta_fan = pr.shoot_rays(source_depth, source_range, eig_angles_hat, receiver_range, num_range_save, envi_z200_r3000)

In [ ]:
eigs_z200_r3000.ray_id[0]

In [ ]:
xi1 = eigs_z200_r3000.zs[0] - eig_dtheta_fan.zs
xi1 = xi1 / xi1[:,-1][:,None]

In [ ]:
for ray_id in ['7.0', '-7.0', '-6.0', '6.0']:
    eig_idx = np.where(eigs_z200_r3000.ray_id[0] == ray_id)[0]
    plt.plot(xi1[eig_idx].T, label=ray_id)

plt.legend()

In [ ]:
eigs_z200_r3000.plot_ducted()

In [ ]:
print(np.where(eigs_z200_r3000.ray_id[0] == '7.0'))
print(np.where(eigs_z300_r5000.ray_id[0] == '7.0'))
print(np.where(eigs_z400_r5000.ray_id[0] == '7.0'))
print(np.where(eigs_z500_r5000.ray_id[0] == '7.0'))

In [ ]:
eigs_z500_r5000.plot_angle_time()
plt.xlim([100,102])

In [ ]:
rays_z200_r3000.plot_time_front(c='C0', add_colorbar=False)
rays_z300_r5000.plot_time_front(c='C1', add_colorbar=False)
rays_z400_r5000.plot_time_front(c='C2', add_colorbar=False)
rays_z500_r5000.plot_time_front(c='C3', add_colorbar=False)
rays_z800_r10000.plot_time_front(c='C4', add_colorbar=False)

plt.xlim([100,102])
plt.ylim([-2500,0])

In [ ]:
eigs_z200_r3000.plot_ducted(c='C0')
eigs_z300_r5000.plot_ducted(c='C1')
eigs_z400_r5000.plot_ducted(c='C2')
eigs_z500_r5000.plot_ducted(c='C3')
eigs_z800_r10000.plot_ducted(c='C4')

plt.ylim([5000,0])

## Sweep Smoothing Variables

In [ ]:
from nescafe23_fluctuations import tools
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

In [ ]:
z_smooth_values = np.array([100, 300, 500, 1000])
r_smooth_values = np.array([3000, 8000])

zz_smooths, rr_smooths = np.meshgrid(z_smooth_values, r_smooth_values)
zz_smooths = zz_smooths.flatten()
rr_smooths = rr_smooths.flatten()


In [ ]:
rvs = []

for k in tqdm(range(len(zz_smooths))):   
    z_smooth = zz_smooths[k]
    r_smooth = rr_smooths[k]
    envi_field = tools.create_2D_ocean_slice(z_smooth=z_smooth, r_smooth=r_smooth)
    rays, eigs, rays_back, eigs_back, envi, envi_back, geo = tools.run_rays(envi_field, 3000)
    ray_run = {
        "rays": rays,
        "eigs": eigs,
        "rays_back": rays_back,
        "eigs_back": eigs_back,
        "envi": envi,
        "envi_back": envi_back,
        "geo": geo,
    }
    
    rvs.append(tools.compute_n23_ray_variables(envi_field = envi_field, ray_run=ray_run))

In [ ]:
Ds = {}

for ray_id in ['+7','-7','+6','-6']:
    Ds[ray_id] = []
    for k in range(len(rvs)):
        Ds[ray_id].append(tools.compute_phase_structure_function(rvs[k][ray_id][0], np.linspace(0,300,100)))

In [ ]:
for D in Ds['-7']:
    plt.plot(D)

In [ ]:
D = tools.compute_phase_structure_function(rvs['-6'][0], np.linspace(0,300,100))
plt.plot(np.exp(-D/2))

D = tools.compute_phase_structure_function(rvs['+6'][0], np.linspace(0,300,100))
plt.plot(np.exp(-D/2))

D = tools.compute_phase_structure_function(rvs['-7'][0], np.linspace(0,300,100))
plt.plot(np.exp(-D/2))

D = tools.compute_phase_structure_function(rvs['+7'][0], np.linspace(0,300,100))
plt.plot(np.exp(-D/2))
